# Bronze → Silver → Gold on Microsoft Fabric

End-to-end lakehouse pipeline: raw CSV files (Bronze) → rule-based cleansing with reject handling and row-count reconciliation (Silver) → star schema for a Direct Lake semantic model (Gold).

Run in a Fabric notebook with the lakehouse attached as default. Upload the three CSV files from `data/bronze/` to `Files/bronze/` first.

In [ ]:
import pyspark.sql.functions as F

for f in mssparkutils.fs.ls("Files/bronze"):
    print(f.name, f.size)

## Silver – cleansing, typing, reject rules, reconciliation

In [ ]:
def load(name):
    df = spark.read.option("header", True).csv(f"Files/bronze/{name}.csv")
    return df.select([F.trim(F.col(c)).alias(c) for c in df.columns])

o, p, c = load("orders"), load("products"), load("customers")

# master data: type, normalise, dedupe on key
p = (p.withColumn("list_price", F.col("list_price").cast("decimal(10,2)"))
       .withColumn("product_name", F.initcap("product_name"))
       .dropDuplicates(["product_id"]))
c = (c.filter(F.col("customer_name") != "")
       .dropDuplicates(["customer_id"]))

# transactions: parse both date formats, type, normalise currency, exact-duplicate removal
o = (o.withColumn("order_date",
        F.coalesce(F.to_date("order_date", "yyyy-MM-dd"), F.to_date("order_date", "dd.MM.yyyy")))
      .withColumn("quantity",   F.col("quantity").cast("int"))
      .withColumn("unit_price", F.col("unit_price").cast("decimal(10,2)"))
      .withColumn("currency",   F.upper("currency"))
      .withColumn("load_ts",    F.current_timestamp())
      .dropDuplicates())

# reject rules – each row gets at most one reason
valid_products  = [r[0] for r in p.select("product_id").collect()]
valid_customers = [r[0] for r in c.select("customer_id").collect()]
o = o.withColumn("reject_reason",
        F.when(F.col("order_id") == "", "missing_order_id")
         .when(F.col("customer_id") == "", "missing_customer")
         .when(F.col("order_date").isNull(), "invalid_date")
         .when(F.col("quantity") <= 0, "non_positive_qty")
         .when(~F.col("product_id").isin(valid_products), "unknown_product")
         .when(~F.col("customer_id").isin(valid_customers), "unknown_customer"))

good    = o.filter(F.col("reject_reason").isNull()).drop("reject_reason")
rejects = o.filter(F.col("reject_reason").isNotNull())

good.write.mode("overwrite").format("delta").saveAsTable("silver_orders")
rejects.write.mode("overwrite").format("delta").saveAsTable("silver_orders_rejects")
p.write.mode("overwrite").format("delta").saveAsTable("silver_products")
c.write.mode("overwrite").format("delta").saveAsTable("silver_customers")

# reconciliation: raw = good + rejects + duplicates removed
raw_cnt = spark.read.option("header", True).csv("Files/bronze/orders.csv").count()
print(f"raw {raw_cnt} = good {good.count()} + rejects {rejects.count()} + dedup {raw_cnt - good.count() - rejects.count()}")
display(rejects.groupBy("reject_reason").count())

## Gold – star schema (one fact, three dimensions, contiguous calendar)

In [ ]:
spark.sql("""
CREATE OR REPLACE TABLE gold_dim_product AS
SELECT product_id, product_name, category, list_price FROM silver_products""")

spark.sql("""
CREATE OR REPLACE TABLE gold_dim_customer AS
SELECT customer_id, customer_name, country, channel FROM silver_customers""")

spark.sql("""
CREATE OR REPLACE TABLE gold_dim_date AS
SELECT d AS date_key,
       year(d) AS year, month(d) AS month, quarter(d) AS quarter,
       date_format(d, 'yyyy-MM') AS year_month,
       date_format(d, 'MMM') AS month_name,
       dayofweek(d) AS weekday
FROM (SELECT explode(sequence(DATE'2025-01-01', DATE'2026-12-31', INTERVAL 1 DAY)) AS d)""")

spark.sql("""
CREATE OR REPLACE TABLE gold_fact_sales AS
SELECT o.order_id, o.order_date AS date_key, o.customer_id, o.product_id,
       o.quantity, o.unit_price,
       CAST(o.quantity * o.unit_price AS decimal(12,2)) AS revenue,
       CAST(o.quantity * (p.list_price - o.unit_price) AS decimal(12,2)) AS discount_amount
FROM silver_orders o
LEFT JOIN silver_products p ON o.product_id = p.product_id""")

display(spark.sql("""
SELECT year_month, SUM(revenue) AS revenue, COUNT(*) AS orders
FROM gold_fact_sales f JOIN gold_dim_date d ON f.date_key = d.date_key
GROUP BY year_month ORDER BY year_month"""))